In [1]:
import numpy as np
from sklearn.utils import shuffle

### 3.3 単純パーセプトロン

In [ ]:
class SimplePerceptron(object):
    """
    単純パーセプトロン
    """

    def __init__(self, input_dim: int) -> None:
        self.input_dim = input_dim
        self.w = np.random.normal(size=(input_dim,))
        self.b = 0.0

    def _step(self, x: np.ndarray) -> np.ndarray:
        return 1 * (x > 0)

    def forward(self, x: np.ndarray) -> np.ndarray:
        y = self._step(np.matmul(self.w, x) + self.b)
        return y

    def compute_deltas(self, x: np.ndarray, t: float) -> tuple[np.ndarray, float]:
        y = self.forward(x)
        delta = y - t
        dw = delta * x
        db = delta
        return dw, db

In [ ]:
"""
1. データの準備
"""

np.random.seed(123)

d = 2
N = 20

mean = 5

# 第二項は各列の平均
x1 = np.random.randn(N // 2, d) + np.array([0, 0])
x2 = np.random.randn(N // 2, d) + np.array([mean, mean])

t1 = np.zeros(N // 2)
t2 = np.ones(N // 2)

# 行方向につなげてる
x = np.concatenate((x1, x2), axis=0)
t = np.concatenate((t1, t2))

"""
2. モデルの構築
"""
model = SimplePerceptron(input_dim=d)
"""
3. モデルの学習
"""


def compute_loss(dw: np.ndarray, db: float) -> np.ndarray:
    # 誤差が0かをみてるだけ
    return all(dw == 0) * (db == 0)


def train_step(
    model: SimplePerceptron, x: np.ndarray, t: float
) -> tuple[bool, SimplePerceptron]:
    # 与えられたデータから損失関数計算してパラメータ計算
    dw, db = model.compute_deltas(x, t)
    loss = compute_loss(dw, db)
    model.w = model.w - dw
    model.b = model.b - db
    return loss, model


while True:
    classified = True
    for i in range(N):
        loss, model = train_step(model, x[i], t[i])
        classified *= loss
    if classified:
        break
"""
4. モデルの評価
"""
display(f"w: {model.w}")
display(f"b: {model.b}")

'w: [2.22951939 2.96727454]'

'b: -13.0'

In [23]:
display(f"(0, 0) => {model.forward([0, 0])}")
display(f"(5, 5) => {model.forward([5, 5])}")

'(0, 0) => 0'

'(5, 5) => 1'

### 3.4 ロジスティック回帰

In [ ]:
class LogisticRegression(object):
    """
    ロジスティック回帰
    """

    def __init__(self, input_dim: int) -> None:
        self.input_dim = input_dim
        self.w = np.random.normal(size=(input_dim,))
        self.b = 0.0

    def __call__(self, x: np.ndarray) -> np.ndarray:
        return self.forward(x)

    def forward(self, x: np.ndarray) -> np.ndarray:
        return self._sigmoid(np.matmul(x, self.w) + self.b)

    def compute_gradients(
        self, x: np.ndarray, t: np.ndarray
    ) -> tuple[np.ndarray, np.ndarray]:
        y = self.forward(x)
        delta = y - t
        dw = np.matmul(x.T, delta)
        db = np.matmul(np.ones(x.shape[0]), delta)
        return dw, db

    def _sigmoid(self, x: np.ndarray) -> np.ndarray:
        return 1 / (1 + np.exp(-x))

In [ ]:
np.random.seed(123)

"""
1. データの準備
"""
# OR
x = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
t = np.array([0, 1, 1, 1])
"""
2. モデルの構築
"""
model = LogisticRegression(input_dim=2)

"""
3. モデルの学習
"""


def compute_loss(t: np.ndarray, y: np.ndarray) -> float:
    return (-t * np.log(y) - (1 - t) * np.log(1 - y)).sum()


def train_step(model: LogisticRegression, x: np.ndarray, t: np.ndarray) -> float:
    dw, db = model.compute_gradients(x, t)
    model.w = model.w - 0.1 * dw
    model.b = model.b - 0.1 * db
    loss = compute_loss(t, model(x))
    return loss


epochs = 100
for epoch in range(epochs):
    train_loss = train_step(model, x, t)  # バッチ学習
    if epoch % 10 == 0 or epoch == epochs - 1:
        display(f"epoch: {epoch + 1}, loss: {train_loss:.3f}")

"""
4. モデルの評価
"""
for input in x:
    display(f"{input} => {model(input):.3f}")

'epoch: 1, loss: 2.812'

'epoch: 11, loss: 1.782'

'epoch: 21, loss: 1.514'

'epoch: 31, loss: 1.339'

'epoch: 41, loss: 1.201'

...

'epoch: 100, loss: 0.735'

'[0 0] => 0.361'

'[0 1] => 0.900'

'[1 0] => 0.844'

'[1 1] => 0.988'

### 3.5 多クラスロジスティック回帰

In [ ]:
class MultiClassLogisticRegression(object):
    """
    (多クラス)ロジスティック回帰
    """

    def __init__(self, input_dim: int, outpud_dim: int) -> None:
        self.input_dim = input_dim
        self.W = np.random.normal(size=(input_dim, outpud_dim))
        self.b = np.zeros(outpud_dim)

    def _softmax(self, x: np.ndarray) -> np.ndarray:
        return np.exp(x) / np.sum(np.exp(x), axis=1, keepdims=True)

    def forward(self, x: np.ndarray) -> np.ndarray:
        return self._softmax(np.matmul(x, self.W) + self.b)

    def __call__(self, x: np.ndarray):
        return self.forward(x)

    def compute_gradients(
        self, x: np.ndarray, t: np.ndarray
    ) -> tuple[np.ndarray, np.ndarray]:
        y = self.forward(x)
        delta = y - t
        dW = np.matmul(x.T, delta)
        db = np.matmul(np.ones(x.shape[0]), delta)
        return dW, db

In [ ]:
"""
1. データの準備
"""

M = 2  # 入力データの次元(入力特徴量の数)
K = 3  # クラス数
n = 100  # クラスごとのデータ数
N = n * K  # 全データ数

x1 = np.random.randn(n, M) + np.array([0, 10])  # K=1では特徴量の平均は0と10
x2 = np.random.randn(n, M) + np.array([5, 5])  # K=1では特徴量の平均は5と5
x3 = np.random.randn(n, M) + np.array([10, 0])  # K=1では特徴量の平均は10と0
t1 = np.array([[1, 0, 0] for i in range(n)])
t2 = np.array([[0, 1, 0] for i in range(n)])
t3 = np.array([[0, 0, 1] for i in range(n)])

x = np.concatenate((x1, x2, x3), axis=0)
t = np.concatenate((t1, t2, t3), axis=0)

"""
2. モデルの構築
"""
model = MultiClassLogisticRegression(input_dim=M, outpud_dim=K)

"""
3. モデルの学習
"""


def compute_loss(t: np.ndarray, y: np.ndarray) -> float:
    return (-t * np.log(y)).sum(axis=1).mean()


def train_step(
    model: MultiClassLogisticRegression, x: np.ndarray, t: np.ndarray
) -> float:
    dW, db = model.compute_gradients(x, t)
    model.W = model.W - 0.1 * dW
    model.b = model.b - 0.1 * db
    loss = compute_loss(t, model(x))
    return loss


epochs = 10
batch_size = 50
n_batches = x.shape[0] // batch_size
for epoch in range(epochs):
    train_loss = 0
    x_, t_ = shuffle(x, t)
    for n_batch in range(n_batches):
        start = n_batch * batch_size
        end = start + batch_size
        train_loss += train_step(
            model, x_[start:end], t_[start:end]
        )  # ミニバッチはデータ(シャッフル済)を分割したものを順番に指定している
    if epoch % 10 == 0 or epoch == epochs - 1:
        display(f"epoch: {epoch + 1}, loss: {train_loss:.3f}")

"""
4. モデルの評価
"""
x_, t_ = shuffle(x, t)
preds = model(x_[0:5])
classified = np.argmax(t_[0:5], axis=1) == np.argmax(preds[0:5], axis=1)
display(f"Prediction matched: {classified}")

'epoch: 1, loss: 112.044'

'epoch: 10, loss: 0.000'

'Prediction matched: [ True  True  True  True  True]'